In [2]:
library(readxl)
library(dplyr)
library(tidyr)
library(ggplot2)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'ggplot2' was built under R version 4.3.3"


# 1. Data

## Main Dataset - BLS LAUS Data

In [13]:
df <- read_excel(
  "../data/ststdsadata.xlsx",
  skip     = 8,
  col_names = FALSE
)

fips_to_abbr <- c(
  "01"="AL","02"="AK","04"="AZ","05"="AR","06"="CA",
  "08"="CO","09"="CT","10"="DE","12"="FL","13"="GA",
  "15"="HI","16"="ID","17"="IL","18"="IN","19"="IA",
  "20"="KS","21"="KY","22"="LA","23"="ME","24"="MD",
  "25"="MA","26"="MI","27"="MN","28"="MS","29"="MO",
  "30"="MT","31"="NE","32"="NV","33"="NH","34"="NJ",
  "35"="NM","36"="NY","37"="NC","38"="ND","39"="OH",
  "40"="OK","41"="OR","42"="PA","44"="RI","45"="SC",
  "46"="SD","47"="TN","48"="TX","49"="UT","50"="VT",
  "51"="VA","53"="WA","54"="WV","55"="WI","56"="WY"
)

df <- df  %>%
  select(fips = 1, state_name = 2, year = 3, month = 4, pop=6, unemp_pop = 10, ur = 11) %>%
  filter(
    !is.na(fips),
    !is.na(ur),
    fips %in% names(fips_to_abbr)
  ) %>%
  mutate(
    state = fips_to_abbr[fips],
    year  = as.integer(year),
    month = as.integer(month),
    pop   = as.integer(pop),
    unemp_pop = as.integer(unemp_pop),
    ur    = as.numeric(ur),
    date  = as.Date(paste(year, month, "01", sep = "-"))
  ) %>%
  select(state, date, population = pop, unemployed_population = unemp_pop, unemployment_rate = ur) %>% 
  filter(date >= as.Date("2000-01-01"), date <= as.Date("2025-12-01")) %>%
  arrange(state, date)
    

Warning message:
"Expecting numeric in E31650 / R31650C5: got '<e2><80><93>'"
Warning message:
"Expecting numeric in F31650 / R31650C6: got '<e2><80><93>'"
Warning message:
"Expecting numeric in G31650 / R31650C7: got '<e2><80><93>'"
Warning message:
"Expecting numeric in H31650 / R31650C8: got '<e2><80><93>'"
Warning message:
"Expecting numeric in I31650 / R31650C9: got '<e2><80><93>'"
Warning message:
"Expecting numeric in J31650 / R31650C10: got '<e2><80><93>'"
Warning message:
"Expecting numeric in K31650 / R31650C11: got '<e2><80><93>'"
Warning message:
"Expecting numeric in E31651 / R31651C5: got '<e2><80><93>'"
Warning message:
"Expecting numeric in F31651 / R31651C6: got '<e2><80><93>'"
Warning message:
"Expecting numeric in G31651 / R31651C7: got '<e2><80><93>'"
Warning message:
"Expecting numeric in H31651 / R31651C8: got '<e2><80><93>'"
Warning message:
"Expecting numeric in I31651 / R31651C9: got '<e2><80><93>'"
Warning message:
"Expecting numeric in J31651 / R31651C10: got

quick data quality check

In [11]:
df %>%
  group_by(state) %>%
  summarise(
    first = min(date),
    last  = max(date),
    n     = n()
  ) %>%
  filter(n != 312) %>%
  arrange(n)

state,first,last,n
<chr>,<date>,<date>,<int>
AK,2000-01-01,2025-12-01,311
AL,2000-01-01,2025-12-01,311
AR,2000-01-01,2025-12-01,311
AZ,2000-01-01,2025-12-01,311
CA,2000-01-01,2025-12-01,311
CO,2000-01-01,2025-12-01,311
CT,2000-01-01,2025-12-01,311
DE,2000-01-01,2025-12-01,311
FL,2000-01-01,2025-12-01,311


### Showcase data

In [14]:
head(df)
tail(df)

nrow(df) # should be 50*26*12 - 50 = 15500 rows since 2025-10-01 is missing for all states

state,date,population,unemployed_population,unemployment_rate
<chr>,<date>,<int>,<int>,<dbl>
AK,2000-01-01,318726,19623,6.2
AK,2000-02-01,319118,19758,6.2
AK,2000-03-01,319507,19923,6.2
AK,2000-04-01,319813,20106,6.3
AK,2000-05-01,319969,20266,6.3
AK,2000-06-01,319938,20354,6.4


state,date,population,unemployed_population,unemployment_rate
<chr>,<date>,<int>,<int>,<dbl>
WY,2025-06-01,291960,9696,3.3
WY,2025-07-01,291172,9548,3.3
WY,2025-08-01,290472,9384,3.2
WY,2025-09-01,290192,9472,3.3
WY,2025-11-01,289848,9724,3.4
WY,2025-12-01,289997,9914,3.4


[1] 15550

## National Covariate Data

### Federal Funds Rate

In [6]:
fedfunds_df <- read.csv("../data/FEDFUNDS.csv") %>%
  mutate(
    date     = as.Date(observation_date),
    fed_rate = as.numeric(FEDFUNDS)
  ) %>%
  select(date, fed_rate) |>
  filter(date >= as.Date("2000-01-01"),
         date <= as.Date("2025-12-01"))

head(fedfunds_df)
tail(fedfunds_df)

# 312 rows = 26 years * 12 months

,date,fed_rate
,<date>,<dbl>
1,2000-01-01,5.45
2,2000-02-01,5.73
3,2000-03-01,5.85
4,2000-04-01,6.02
5,2000-05-01,6.27
6,2000-06-01,6.53


,date,fed_rate
,<date>,<dbl>
307,2025-07-01,4.33
308,2025-08-01,4.33
309,2025-09-01,4.22
310,2025-10-01,4.09
311,2025-11-01,3.88
312,2025-12-01,3.72


###  Real GDP growth (quarterly -> interpolate to monthly)

In [8]:
gdp_raw <- read.csv("../data//REALGDP.csv") %>%
  mutate(
    date       = as.Date(observation_date),
    gdp_growth = as.numeric(A191RL1Q225SBEA)
  ) %>%
  select(date, gdp_growth) %>%
  filter(!is.na(gdp_growth))

# Method of interpolation: each month gets the value of its quarter
monthly_dates <- seq(as.Date("2000-01-01"),
                     as.Date("2025-12-01"), by = "month")
                     
gdp_df <- data.frame(date = monthly_dates) %>%
  mutate(
    gdp_growth = gdp_raw$gdp_growth[
      findInterval(date, gdp_raw$date)
    ]
  )

head(gdp_df)
tail(gdp_df)

,date,gdp_growth
,<date>,<dbl>
1,2000-01-01,1.5
2,2000-02-01,1.5
3,2000-03-01,1.5
4,2000-04-01,7.5
5,2000-05-01,7.5
6,2000-06-01,7.5


,date,gdp_growth
,<date>,<dbl>
307,2025-07-01,4.4
308,2025-08-01,4.4
309,2025-09-01,4.4
310,2025-10-01,0.7
311,2025-11-01,0.7
312,2025-12-01,0.7


## Export all cleaned data as CSVs for reproducable use

In [9]:
write.csv(df, "../cleaned_data/bls_data.csv", row.names = FALSE)
write.csv(fedfunds_df, "../cleaned_data/fedfunds_data.csv", row.names = FALSE)
write.csv(gdp_df, "../cleaned_data/gdp_data.csv", row.names = FALSE)